In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
BATCH_SIZE = 64 # how many independent sequences will we process in parallel?
BLOCK_SIZE = 256 # what is the maximum context length for predictions?
MAX_TRAIN_ITERS = 5_000
EVAL_INTERVAL = 500
LR = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
eval_iters = 200
N_EMB = 384
N_HEADS = 6
FF_RESID_MULTIPLIER = 4
N_LAYERS = 2
DROPOUT = 0.2 # see paper. shuts off neurons randomly during training
# to improve generalization and reduce overfitting
# applied to activation functions
# simply sets P random values to zero
# ------------

torch.manual_seed(1337)

cpu


In [2]:
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
VOCAB_SIZE = len(chars)

# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# Tokenize: raw text to a sequence of integers
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string
# More advanced tokenizers return shorter lists (we return i for each char), but with higher max i

In [3]:
data = torch.tensor(encode(text), dtype=torch.long)
len(data)

1115394

In [11]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]
train_data.shape

torch.Size([1003854])

In [7]:
# norm_dir = 1 turns BatchNorm -> LayerNorm

class LayerNorm(nn.Module):
  
  def __init__(self, dim, norm_dir=1, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    self.norm_dir = norm_dir
  
  def __call__(self, x):
    """
    x is 32x100 (output of Linear layer)
    wavenet x is 32x4x200
    """
    
    xmean = x.mean(self.norm_dir, keepdim=True) # batch mean - across axis 0 and 1, shape is 1xBATCH_SIZE
    xvar = x.var(self.norm_dir, keepdim=True) # batch variance - across axis 0 and 1
    
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

# Appendix
torch.manual_seed(1337)

m = LayerNorm(5, norm_dir=0)
x = torch.randn(2, 5)
print('x:', x)
avg = x.mean(0, keepdim=True)
v = x.var(0, keepdim=True)
print('avg:', avg)
print('var:', v)
y = (x - avg) / torch.sqrt(v + 1e-5)
print(f"{y=}")
n = m(x)
print(f"{n=}")

print("\n\n")
layer_norm = LayerNorm(5, norm_dir=1)
avg = x.mean(1, keepdim=True)
v = x.var(1, keepdim=True)
print('avg:', avg)
print('var:', v)
y = (x - avg) / torch.sqrt(v + 1e-5)
print(y)
n = layer_norm(x)
print(n)


x: tensor([[-2.0260, -2.0655, -1.2054, -0.9122, -1.2502],
        [ 0.8032, -0.2071,  0.0544,  0.1378, -0.3889]])
avg: tensor([[-0.6114, -1.1363, -0.5755, -0.3872, -0.8196]])
var: tensor([[4.0023, 1.7268, 0.7935, 0.5513, 0.3709]])
y=tensor([[-0.7071, -0.7071, -0.7071, -0.7071, -0.7071],
        [ 0.7071,  0.7071,  0.7071,  0.7071,  0.7071]])
n=tensor([[-0.7071, -0.7071, -0.7071, -0.7071, -0.7071],
        [ 0.7071,  0.7071,  0.7071,  0.7071,  0.7071]])



avg: tensor([[-1.4919],
        [ 0.0799]])
var: tensor([[0.2727],
        [0.2073]])
tensor([[-1.0228, -1.0985,  0.5486,  1.1099,  0.4627],
        [ 1.5885, -0.6303, -0.0559,  0.1273, -1.0295]])
tensor([[-1.0228, -1.0985,  0.5486,  1.1099,  0.4627],
        [ 1.5885, -0.6303, -0.0559,  0.1273, -1.0295]])


In [8]:

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ See Appendix """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMB, head_size, bias=False)
        self.query = nn.Linear(N_EMB, head_size, bias=False)
        self.value = nn.Linear(N_EMB, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))

        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape # BATCH_SIZE, BLOCK_SIZE, N_EMB

        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        v = self.value(x) # (B,T,hs)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

        # for residual connections we need a projection layer to combine the outputs of all the heads
        self.proj = nn.Linear(N_EMB, N_EMB)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # concat over Channel (embeds)
        out = self.proj(out)
        out = self.dropout(out)
        return out # (B, T, head_size * num_heads)

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd, ff_multiplier):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, ff_multiplier*n_embd),
            nn.ReLU(),
            nn.Linear(ff_multiplier*n_embd, n_embd),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd, FF_RESID_MULTIPLIER)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # residual connections
        # we fork off, we do some computation, 
        # and then we add it back to the original x

        # note we notmalize x BEFORE we feed it into the self attention and feed forward layers
        x = x + self.sa(self.ln1(x)) # self attention part
        x = x + self.ffwd(self.ln2(x)) # computation part
        return x
    
# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        # eg 32x3 in
        # 32x3x10 out
        # for each of 3 chars we pull 10 embs
        self.token_embedding_table = nn.Embedding(VOCAB_SIZE, N_EMB)

        # Emb position
        # Each position, from 0 to BLOCK_SIZE-1, will get its own embedding vector
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMB)

        self.blocks = nn.Sequential(*[Block(N_EMB, N_HEADS) for _ in range(N_LAYERS)]+[nn.LayerNorm(N_EMB)])
        
        self.ll_1 = nn.Linear(N_EMB, VOCAB_SIZE)

    def forward(self, x, targets=None):
        """
        Forward pass + Loss

        """

        # x and targets are both (B,T) tensor of integers
        # x is BatchSize x Time (block size)

        B, T = x.shape

        te = self.token_embedding_table(x) # (B,T,C) Batch(size=4) Time(=Block=8) Channel(=emb_dim=vocab size=65)
        pe = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) Time(=Block=8) Channel(=emb_dim=vocab size=65)

        # note for each batch B
        # we add pe to te
        # because pe doesn't change with batch
        e = te + pe # (B,T,C) add the position information to the token information

        x = self.blocks(e) # (B,T,C)

        # print(x.shape)
        
        logits = self.ll_1(x) # (B,T,VOCAB_SIZE)

        if targets is None:
            loss = None
        else:
            # pyTorch cross_entropy want B, C, T but we have B, T, C so need to reshape
            # Batch
            # Time
            # Channel
            # print(logits.shape)
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # preserve C, stretch out B and T into single dim
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        # idx is (BATCH_SIZE, BLOCK_SIZE) array of indices in the current context

        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -BLOCK_SIZE:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [87]:
model = BigramLanguageModel()
# m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)


In [ ]:
for iter in range(MAX_TRAIN_ITERS):
# for iter in range(1):

    # every once in a while evaluate the loss on train and val sets
    if iter % EVAL_INTERVAL == 0:
        losses = estimate_loss(model)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



In [82]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=100)[0].tolist()))


G:
Can teed dord'ld gothamd,
Itest: ofrfeast, re ombs,slam?

LINARIERY:
S of youchwn.

WAINesperreul
